# SmolLM2-360M-Instruct â€” DRAG Fine-tuning (SFT)

**Model**: `HuggingFaceTB/SmolLM2-360M-Instruct`  
**Method**: Supervised Fine-tuning (SFT) via TRL `SFTTrainer`  
**Dataset**: ChatML JSONL produced by `scripts/gen-training-data.mjs`  
**Target runtime**: T4 GPU (~2â€“4 hours)

## Steps
1. Run the install cell
2. Upload `training_data.jsonl` when prompted
3. Run remaining cells in order
4. Download `checkpoint.zip`
5. Run `export_onnx.py` locally (or continue in Colab) to produce ONNX int8

In [5]:
# Cell 1 â€” Install dependencies
!pip install -q trl transformers datasets accelerate
!pip install -q optimum[exporters] onnx onnxruntime
print('Installation complete.')

Installation complete.


In [14]:
# Cell 2 â€” Load training_data.jsonl (local path or Colab upload)
import os
import shutil

jsonl_path = 'training_data.jsonl'

# --- Option A: Google Drive (preferred when running in Colab) ---
if os.getcwd().startswith('/content') and not os.path.exists(jsonl_path):
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        DRIVE_PATH = '/content/drive/MyDrive/ColabDrive/Portfolio/training_data.jsonl'
        if os.path.exists(DRIVE_PATH):
            shutil.copy(DRIVE_PATH, jsonl_path)
            print(f'Copied from Google Drive: {DRIVE_PATH} -> {jsonl_path}')
        else:
            print(f'WARNING: {DRIVE_PATH} not found on Drive.')
            print('Upload training_data.jsonl to MyDrive/ColabDrive/Portfolio/ and rerun,')
            print('or use the Colab upload widget below.')
    except ImportError:
        pass  # not in Colab

# --- Option B: running locally or file already present ---
LOCAL_CANDIDATES = [
    'training_data.jsonl',                                   # CWD
    'scripts/finetune/training_data.jsonl',                  # repo root relative
    os.path.join(os.path.dirname(os.path.abspath('.')),
                 'scripts', 'finetune', 'training_data.jsonl'),
]

if not os.path.exists(jsonl_path):
    for candidate in LOCAL_CANDIDATES:
        if os.path.exists(candidate):
            if candidate != jsonl_path:
                shutil.copy(candidate, jsonl_path)
                print(f'Copied {candidate} -> {jsonl_path}')
            else:
                print(f'Found {jsonl_path} in CWD.')
            break

# --- Option C: Colab upload widget (fallback) ---
if not os.path.exists(jsonl_path):
    try:
        from google.colab import files
        print('File not found locally or on Drive â€” launching Colab upload widget...')
        uploaded = files.upload()
        if jsonl_path not in uploaded:
            for name in uploaded:
                if name.endswith('.jsonl'):
                    os.rename(name, jsonl_path)
                    print(f'Renamed {name} -> {jsonl_path}')
                    break
    except ImportError:
        pass  # not in Colab

assert os.path.exists(jsonl_path), (
    f'{jsonl_path} not found.\n'
    'Options:\n'
    '  1. Upload to Google Drive at MyDrive/ColabDrive/Portfolio/training_data.jsonl\n'
    '  2. Place training_data.jsonl next to this notebook or in scripts/finetune/\n'
    '  3. Use the Colab upload widget (Option C above)'
)

with open(jsonl_path) as f:
    line_count = sum(1 for _ in f)
print(f'Training examples: {line_count}')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Copied from Google Drive: /content/drive/MyDrive/ColabDrive/Portfolio/training_data.jsonl -> training_data.jsonl
Training examples: 96


In [15]:
# Cell 3 â€” Load and inspect dataset
import json
from datasets import Dataset

def load_chatml_jsonl(path):
    records = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            records.append(json.loads(line))
    return records

raw_data = load_chatml_jsonl(jsonl_path)
print(f'Loaded {len(raw_data)} records')
print('Sample:', json.dumps(raw_data[0], indent=2))

Loaded 96 records
Sample: {
  "messages": [
    {
      "role": "user",
      "content": "What is Kham's email address?"
    },
    {
      "role": "assistant",
      "content": "<email>"
    }
  ],
  "meta": {
    "sourceChunkIds": [
      "chunk-0"
    ],
    "groundingScore": 1,
    "generatedBy": "llama-3.3-70b-versatile",
    "questionType": "factual"
  }
}


In [19]:
# Cell 4 â€” Tokenize and split
import os
from transformers import AutoTokenizer

# Authenticate with Hugging Face Hub
hf_token = None

# Option A: Colab secret
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        print('Found HF_TOKEN from Colab secret.')
except Exception:
    pass  # not in Colab or secret unavailable

# Option B: environment variable
if not hf_token:
    hf_token = os.environ.get('HF_TOKEN')
    if hf_token:
        print('Found HF_TOKEN from environment variable.')

# Option C: interactive prompt
if not hf_token:
    import getpass
    hf_token = getpass.getpass('Enter your Hugging Face token (leave blank to skip): ').strip() or None
    if hf_token:
        print('Using token from prompt.')

if hf_token:
    from huggingface_hub import login
    login(token=hf_token, add_to_git_credential=False)
    print('Authenticated with Hugging Face Hub.')
else:
    print('WARNING: No HF token provided â€” proceeding unauthenticated (public models only).')

BASE_MODEL = 'HuggingFaceTB/SmolLM2-360M-Instruct'
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token

def format_as_chatml(record):
    text = tokenizer.apply_chat_template(
        record['messages'],
        tokenize=False,
        add_generation_prompt=False
    )
    return {'text': text}

dataset = Dataset.from_list(raw_data)
dataset = dataset.map(format_as_chatml, remove_columns=['messages', 'meta'])

split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split['train']
eval_dataset  = split['test']

print(f'Train: {len(train_dataset)} | Eval: {len(eval_dataset)}')
print('Sample (truncated):', train_dataset[0]['text'][:300])


Using token from prompt.
Authenticated with Hugging Face Hub.


Map:   0%|          | 0/96 [00:00<?, ? examples/s]

Train: 86 | Eval: 10
Sample (truncated): <|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
How many years of experience does Kham have in semiconductor hardware expertise and deploying production AI/ML systems?<|im_end|>
<|im_start|>assistant
14 years of semiconductor hardwar


In [25]:
# Cell 5 â€” Load base model
import torch
from transformers import AutoModelForCausalLM

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# Load in float32 â€” SFTConfig fp16=True handles mixed-precision internally.
# Loading directly in float16 causes "Attempting to unscale FP16 gradients" errors
# because the optimizer receives already-cast FP16 weights instead of FP32 master weights.
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float32,
    device_map='auto'
)
model.config.use_cache = False
print(f'Parameters: {model.num_parameters():,}')


Device: cuda
GPU: Tesla T4


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Parameters: 361,821,120


In [28]:
# Cell 6 â€” Configure SFTTrainer
from trl import SFTTrainer, SFTConfig

CHECKPOINT_DIR = './checkpoint'

sft_config = SFTConfig(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=8,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,  # 10% of total steps
    weight_decay=0.01,
    fp16=(device == 'cuda'),
    logging_steps=20,
    eval_strategy='steps',
    eval_steps=100,
    save_strategy='steps',
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    report_to='none',
    dataset_text_field='text',
    max_length=512
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer
)

print('Trainer configured.')


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/86 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/86 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/86 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

Trainer configured.


In [29]:
# Cell 7 â€” Train (~2â€“4 hours on T4)
print('Starting training...')
train_result = trainer.train()
print(f'Training complete. Loss: {train_result.training_loss:.4f}')

Starting training...


Step,Training Loss,Validation Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training complete. Loss: 1.4661


In [30]:
# Cell 8 — Save model and tokenizer
trainer.save_model(CHECKPOINT_DIR)
tokenizer.save_pretrained(CHECKPOINT_DIR)
print(f'Saved to {CHECKPOINT_DIR}')

import os
for fname in os.listdir(CHECKPOINT_DIR):
    size_mb = os.path.getsize(os.path.join(CHECKPOINT_DIR, fname)) / 1e6
    print(f'  {fname}  ({size_mb:.1f} MB)')

# --- Save checkpoint back to Google Drive ---
try:
    from google.colab import drive
    import shutil

    DRIVE_CHECKPOINT = '/content/drive/MyDrive/ColabDrive/Portfolio/checkpoint'
    if os.path.exists('/content/drive'):
        print('Google Drive already mounted.')
    else:
        drive.mount('/content/drive')

    if os.path.exists(DRIVE_CHECKPOINT):
        shutil.rmtree(DRIVE_CHECKPOINT)
        print(f'Removed existing {DRIVE_CHECKPOINT}')

    shutil.copytree(CHECKPOINT_DIR, DRIVE_CHECKPOINT)
    print(f'Checkpoint saved to Google Drive: {DRIVE_CHECKPOINT}')
except ImportError:
    print('Not running in Colab — skipping Google Drive save.')
except Exception as e:
    print(f'WARNING: Could not save to Google Drive: {e}')


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to ./checkpoint
  model.safetensors  (1447.3 MB)
  tokenizer_config.json  (0.0 MB)
  training_args.bin  (0.0 MB)
  README.md  (0.0 MB)
  tokenizer.json  (3.5 MB)
  generation_config.json  (0.0 MB)
  config.json  (0.0 MB)
  checkpoint-18  (0.0 MB)
  chat_template.jinja  (0.0 MB)
  checkpoint-48  (0.0 MB)
Google Drive already mounted.
Checkpoint saved to Google Drive: /content/drive/MyDrive/ColabDrive/Portfolio/checkpoint


In [31]:
# Cell 9 â€” Quick sanity check
from transformers import pipeline

pipe = pipeline('text-generation', model=CHECKPOINT_DIR, tokenizer=CHECKPOINT_DIR,
                device=0 if device == 'cuda' else -1)
prompt = [{'role': 'user', 'content': "What are Kham's strongest technical skills?"}]
result = pipe(prompt, max_new_tokens=150)
print('Sample response:')
print(result[0]['generated_text'][-1]['content'])

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Sample response:
Kham has strong technical skills in RF/Microwave Communications, Advanced Signal Processing, and Power Electronics.


In [33]:
!pwd


/content


In [32]:
# Cell 10 â€” Download checkpoint for local ONNX export
import shutil

shutil.make_archive('checkpoint', 'zip', '.', 'checkpoint')
print('Created checkpoint.zip')

try:
    from google.colab import files
    files.download('checkpoint.zip')
    print('Downloaded checkpoint.zip')
except ImportError:
    print('Not running in Colab â€” checkpoint.zip saved to current directory.')

print('Next step: unzip and run  python scripts/finetune/export_onnx.py --checkpoint ./checkpoint')


Created checkpoint.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded checkpoint.zip
Next step: unzip and run  python scripts/finetune/export_onnx.py --checkpoint ./checkpoint
